<a href="https://colab.research.google.com/github/gaelhugo/ECAL/blob/master/Audio_classification_with_CLAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install git+https://github.com/gaelhugo/CLAP.git/
!pip install pandas librosa

  Cloning https://github.com/gaelhugo/CLAP.git/ to /tmp/pip-req-build-5e6waf_1
  Running command git clone --filter=blob:none --quiet https://github.com/gaelhugo/CLAP.git/ /tmp/pip-req-build-5e6waf_1
  Resolved https://github.com/gaelhugo/CLAP.git/ to commit 807ccab46f87f194dd1a6482e2636cf96492d12d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


on charge le drive perso

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Importer les modules pour extraire les features avec CLAP



In [3]:
import numpy as np
import librosa
import torch
import laion_clap
import os
import logging

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Load Model

In [15]:
model = laion_clap.CLAP_Module(enable_fusion=False, amodel='HTSAT-base')
print("loading checkpoint")
# set gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# List of available models to download: https://github.com/LAION-AI/CLAP?tab=readme-ov-file#pretrained-models
#model.load_ckpt("/content/drive/MyDrive/AudioCheckpoints/music_speech_audioset_epoch_15_esc_89.98.pt")
model.load_ckpt("/content/drive/MyDrive/AudioCheckpoints/music_audioset_epoch_15_esc_90.14.pt")
print("checkpoint loaded")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


loading checkpoint
checkpoint loaded


Lister et analyser les fichiers audio


In [ ]:
os.chdir("/content/drive/MyDrive/Audios")
all_features = []
all_urls = []
for index,url in enumerate(os.listdir()):
  if url != ".DS_Store" and os.path.isfile(url):
    all_urls.append(url)



#get embeddings from the list
with torch.no_grad():
  audio_embed = model.get_audio_embedding_from_filelist(x = all_urls, use_tensor=True)
  print(audio_embed[:,-20:])
  print(audio_embed.shape)
  final_audio_embed = audio_embed


A lot of audios could be problematic. Use smaller batches

In [16]:
# --- Configuration ---
BATCH_SIZE = 10 # Process 10 files at a time. Adjust this based on your RAM/GPU limit.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(DEVICE) # Ensure the model is on the correct device
# ---------------------

os.chdir("/content/drive/MyDrive/Audios")
all_urls = []
# 1. Collect all valid file URLs
for url in os.listdir():
  if url != ".DS_Store" and os.path.isfile(url):
    all_urls.append(url)

# 2. Process files in batches to manage memory
all_embeddings = [] # List to store embeddings from each batch (as NumPy arrays)

with torch.no_grad():
  for i in range(0, len(all_urls), BATCH_SIZE):
    # Get the slice of URLs for the current batch
    batch_urls = all_urls[i:i + BATCH_SIZE]

    print(f"Processing batch {int(i/BATCH_SIZE) + 1}/{int(len(all_urls)/BATCH_SIZE) + 1} with {len(batch_urls)} files...")

    # Get embeddings for the current batch, ensuring the model is on the right device
    audio_embed_batch = model.get_audio_embedding_from_filelist(
        x=batch_urls,
        use_tensor=True
    ).to(DEVICE) # Ensure the tensor is on the processing device if needed

    # *** CRITICAL STEP FOR RAM/VRAM SAVINGS ***
    # Move the resulting tensor to the CPU and convert it to a NumPy array.
    # This releases the GPU/CUDA memory associated with the batch embeddings immediately.
    audio_embed_numpy = audio_embed_batch.cpu().numpy()

    all_embeddings.append(audio_embed_numpy)

    # Optional: Clear VRAM/RAM after each batch
    del audio_embed_batch
    torch.cuda.empty_cache() # Clear unused GPU memory

# 3. Concatenate all batch embeddings into a single NumPy array
final_audio_embed = np.concatenate(all_embeddings, axis=0)

# 4. Final Verification
print("\n--- Final Results ---")
print(f"Final embeddings shape: {final_audio_embed.shape}")
print("Last 20 dimensions of the first embedding:")
print(final_audio_embed[0, -20:])

Processing batch 1/11 with 10 files...
Processing batch 2/11 with 10 files...
Processing batch 3/11 with 10 files...
Processing batch 4/11 with 10 files...
Processing batch 5/11 with 10 files...
Processing batch 6/11 with 10 files...
Processing batch 7/11 with 10 files...
Processing batch 8/11 with 10 files...
Processing batch 9/11 with 10 files...
Processing batch 10/11 with 10 files...
Processing batch 11/11 with 8 files...

--- Final Results ---
Final embeddings shape: (108, 512)
Last 20 dimensions of the first embedding:
[-0.05909371  0.0124395  -0.04896851 -0.03167645 -0.08234429  0.01820983
  0.04836596 -0.01173826  0.05983587 -0.09465923 -0.03207862  0.00766534
  0.0107333   0.00136988  0.06291676  0.03256932  0.10245828 -0.00063522
  0.00586066  0.05956816]


Save data for future use

In [17]:
import pandas as pd
import numpy as np
import os

output_dir = "/content/drive/MyDrive/Data_audio"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save all_urls as a CSV file
urls_df = pd.DataFrame(all_urls, columns=["audio_url"])
urls_df.to_csv(os.path.join(output_dir, "all_urls.csv"), index=False)
print(f"Saved all_urls to {os.path.join(output_dir, 'all_urls.csv')}")

# Save audio_embed as a NumPy file
numpy_audio_embed = final_audio_embed # final_audio_embed is already a numpy array
np.save(os.path.join(output_dir, "audio_embeddings.npy"), numpy_audio_embed)
print(f"Saved audio_embed to {os.path.join(output_dir, 'audio_embeddings.npy')}")

Saved all_urls to /content/drive/MyDrive/Data_audio/all_urls.csv
Saved audio_embed to /content/drive/MyDrive/Data_audio/audio_embeddings.npy


In [ ]:
def get_text_embedding_with_tensor(text_data):
    text_embed = model.get_text_embedding(text_data, use_tensor=True)
    return text_embed


def get_nearest_audio_with_tensor(audio_embed, text_embed):
    nearest_audio = torch.cdist(audio_embed, text_embed, p=2).argmin(dim=0)
    return nearest_audio

def get_nearest_audio_with_tensor_by_manual_calculation(audio_embed, text_embed):
  # Ensure text_embed is normalized (important for cosine similarity)
  text_embed = text_embed / text_embed.norm(dim=-1, keepdim=True)
  # Calculate cosine similarities (dot product since both are normalized)
  similarities = torch.mm(audio_embed, text_embed.t())
  # Get indices of the 10 nearest neighbors (highest similarities)
  nearest_indices = similarities.topk(4, dim=0).indices
  return nearest_indices


In [ ]:
## TEST
while True:
    text = input("Enter text: ")
    text_embed = get_text_embedding_with_tensor(text)
    nearest_audio = get_nearest_audio_with_tensor(audio_embed, text_embed)
    print("Nearest audio: ", nearest_audio)
    # get the name of the nearest audio
    nearest_audio_name = all_urls[nearest_audio.item()]
    print("Nearest audio name: ", nearest_audio_name)
    #-------------------------------------------------------
    #comparison with manual check
    nearest_audio_manual = get_nearest_audio_with_tensor_by_manual_calculation(audio_embed, text_embed)
    print("Nearest audio manual id: ", nearest_audio_manual)
    #print name
    nearest_audio_manual_name = [all_urls[i.item()] for i in nearest_audio_manual]
    print("Nearest audio manual name: ", nearest_audio_manual_name)



Add Annoy index for faster an maybe more efficient nearest neighbors classification


In [8]:
pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp312-cp312-linux_x86_64.whl size=551811 sha256=e960752c5bfec551873384049f7b607100d56209bd514c9fb61b234baeeaf729
  Stored in directory: /root/.cache/pip/wheels/db/b9/53/a3b2d1fe1743abadddec6aa541294b24fdbc39d7800bc57311
Successfully built annoy


In [19]:
from annoy import AnnoyIndex
import os

# Ensure output_dir is defined
output_dir = "/content/drive/MyDrive/Data_audio"

# Get the dimensionality of the embeddings
f = numpy_audio_embed.shape[1]

# Create an Annoy index, using 'angular' for cosine similarity
t = AnnoyIndex(f, 'angular')  # Using 'angular' for cosine similarity, as it's common for embeddings

for i, vec in enumerate(numpy_audio_embed):
    print(i)
    t.add_item(i, vec)

# Build the index (more trees means higher precision, but slower build time)
t.build(10) # 10 trees

# Save the index to a file
annoy_index_path = os.path.join(output_dir, "audio_embeddings.ann")
t.save(annoy_index_path)

print(f"Annoy index created and saved to {annoy_index_path}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
Annoy index created and saved to /content/drive/MyDrive/Data_audio/audio_embeddings.ann


# Task
Create a Python function to chunk audio files into 10-second segments. Then, integrate this function into the existing audio processing workflow to generate embeddings for these 10-second chunks using the CLAP model, replacing the current full-file processing. Finally, save the chunk metadata (original filename and time range) along with their embeddings and update the Annoy index creation to use these chunk embeddings.

## Define chunking function

### Subtask:
Create a Python function that takes an audio file path, loads the audio, splits it into 10-second chunks, and saves each chunk to a temporary directory with a name indicating the original file and time range (e.g., 'original_file_0-10s.wav'). It should return a list of paths to these new chunk files.


**Reasoning**:
The subtask requires splitting audio files into chunks. The `pydub` library is suitable for this task, so I need to install it first.



In [21]:
pip install pydub
print("pydub installed")

SyntaxError: invalid syntax (ipython-input-3731897483.py, line 1)

**Reasoning**:
The previous command failed because `pip install` was used without the `!` prefix in a code cell, leading to a `SyntaxError`. I need to add `!` to execute it as a shell command.



In [22]:
get_ipython().system('pip install pydub')
print("pydub installed")

pydub installed


**Reasoning**:
Now that `pydub` is installed, I will define the `chunk_audio_file` function as per the subtask instructions, which involves importing necessary modules, creating the output directory, loading the audio, segmenting it into chunks, saving each chunk, and collecting their paths.



In [23]:
from pydub import AudioSegment
import os

def chunk_audio_file(audio_path, chunk_length_ms=10000, output_dir='temp_audio_chunks'):
    os.makedirs(output_dir, exist_ok=True)

    try:
        audio = AudioSegment.from_file(audio_path)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return []

    chunk_paths = []
    base_name = os.path.splitext(os.path.basename(audio_path))[0]

    # Calculate total duration in milliseconds
    total_length_ms = len(audio)

    for i in range(0, total_length_ms, chunk_length_ms):
        start_ms = i
        end_ms = min(i + chunk_length_ms, total_length_ms)

        # Skip chunks smaller than a certain threshold to avoid very short, potentially empty files
        if (end_ms - start_ms) < 1000: # e.g., chunks smaller than 1 second are skipped
            continue

        chunk = audio[start_ms:end_ms]
        start_s = start_ms // 1000
        end_s = end_ms // 1000

        # Create a unique filename for the chunk
        chunk_filename = f'{base_name}_{start_s}-{end_s}s.wav'
        chunk_path = os.path.join(output_dir, chunk_filename)

        try:
            chunk.export(chunk_path, format="wav")
            chunk_paths.append(chunk_path)
        except Exception as e:
            print(f"Error exporting chunk {chunk_filename}: {e}")

    return chunk_paths

print("chunk_audio_file function defined.")

chunk_audio_file function defined.


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


**Reasoning**:
The previously defined `chunk_audio_file` function needs to be updated to return not only the paths to the chunk files but also associated metadata (original filename, start time, and end time) for each chunk. This metadata is required to fulfill the task's requirement of saving chunk metadata along with their embeddings.



In [24]:
from pydub import AudioSegment
import os

def chunk_audio_file(audio_path, chunk_length_ms=10000, output_dir='temp_audio_chunks'):
    os.makedirs(output_dir, exist_ok=True)

    try:
        audio = AudioSegment.from_file(audio_path)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return []

    chunk_data = [] # Changed to store dicts/tuples
    base_name = os.path.splitext(os.path.basename(audio_path))[0]

    # Calculate total duration in milliseconds
    total_length_ms = len(audio)

    for i in range(0, total_length_ms, chunk_length_ms):
        start_ms = i
        end_ms = min(i + chunk_length_ms, total_length_ms)

        # Skip chunks smaller than a certain threshold to avoid very short, potentially empty files
        if (end_ms - start_ms) < 1000: # e.g., chunks smaller than 1 second are skipped
            continue

        chunk = audio[start_ms:end_ms]
        start_s = start_ms // 1000
        end_s = end_ms // 1000

        # Create a unique filename for the chunk
        chunk_filename = f'{base_name}_{start_s}-{end_s}s.wav'
        chunk_path = os.path.join(output_dir, chunk_filename)

        try:
            chunk.export(chunk_path, format="wav")
            # Store chunk path and metadata
            chunk_data.append({
                "chunk_path": chunk_path,
                "original_filename": os.path.basename(audio_path),
                "start_s": start_s,
                "end_s": end_s
            })
        except Exception as e:
            print(f"Error exporting chunk {chunk_filename}: {e}")

    return chunk_data # Return the list of dictionaries

print("chunk_audio_file function redefined to return chunk metadata.")

chunk_audio_file function redefined to return chunk metadata.


## Integrate chunking into main loop

### Subtask:
Modify the existing audio processing loop to call the chunking function for each original audio file. Collect all the paths of the generated audio chunks and their corresponding original filenames and time ranges. This will replace the direct use of 'all_urls'.


**Reasoning**:
The subtask requires modifying the existing audio processing loop to chunk original audio files and collect metadata. This step will implement the loop, calling the `chunk_audio_file` function for each original audio file and populating `all_chunk_metadata` and `all_chunk_paths` as specified in the instructions.



In [25]:
import os

# Set the current working directory back to where original audio files are located
os.chdir("/content/drive/MyDrive/Audios")
current_audio_directory = os.getcwd()

# Initialize lists to store chunk metadata and paths
all_chunk_metadata = []
all_chunk_paths = []

# Loop through original audio files and chunk them
# 'all_urls' is already defined from previous steps and contains the filenames
for original_audio_filename in all_urls:
    original_audio_path = os.path.join(current_audio_directory, original_audio_filename)

    print(f"Chunking audio file: {original_audio_filename}")
    chunked_data_for_file = chunk_audio_file(original_audio_path)

    for chunk_info in chunked_data_for_file:
        all_chunk_metadata.append(chunk_info)
        all_chunk_paths.append(chunk_info["chunk_path"])

print(f"Total original files processed: {len(all_urls)}")
print(f"Total chunks generated: {len(all_chunk_paths)}")
print("First 5 chunk paths:")
for i in range(min(5, len(all_chunk_paths))):
    print(all_chunk_paths[i])
print("First 5 chunk metadata entries:")
for i in range(min(5, len(all_chunk_metadata))):
    print(all_chunk_metadata[i])

Chunking audio file: rumba.mp3
Chunking audio file: fonk.mp3
Chunking audio file: balletz.mp3
Chunking audio file: Zaira_VO.mp3
Chunking audio file: Kalabrese - Purple Rose.mp3
Chunking audio file: Karasu (Crowdpleaser remix).mp3
Chunking audio file: KANANASKIS (feat. Phoene Somsavath).mp3
Chunking audio file: Kalaha Moon - Queen Of The Highway (Original Mix).mp3
Chunking audio file: kalupke - Devin The Deviant.mp3
Chunking audio file: Keep Schtum - Guest Mix for Disco Outcasts (Feb '11).mp3
Chunking audio file: Keith Kenniff - All I Needed.mp3
Chunking audio file: Kellerkind And Jake The Rapper - Projection.mp3
Chunking audio file: Kiln - Airplaneshadows.mp3
Chunking audio file: Kim Carroll - The Colony� As it Is (Soundtrack).mp3
Chunking audio file: Kindisch Presents  Kindisch Steps V (Minimix).mp3
Chunking audio file: Kleintierschaukel - Kidz Of Tibet (feat. Badau).mp3
Chunking audio file: Kollektiv Turmstrasse � Harry Klein Podcast 54.mp3
Chunking audio file: Kollektiv Turmstrasse-

**Reasoning**:
The subtask requires generating embeddings for the newly created audio chunks using the CLAP model. This involves adapting the existing batch processing loop to use `all_chunk_paths` instead of `all_urls`.



In [26]:
BATCH_SIZE = 10 # Process 10 files at a time. Adjust this based on your RAM/GPU limit.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(DEVICE) # Ensure the model is on the correct device

# Process files in batches to manage memory, using the chunked audio paths
all_chunk_embeddings = [] # List to store embeddings from each batch (as NumPy arrays)

with torch.no_grad():
  for i in range(0, len(all_chunk_paths), BATCH_SIZE):
    # Get the slice of chunk paths for the current batch
    batch_chunk_paths = all_chunk_paths[i:i + BATCH_SIZE]

    print(f"Processing chunk batch {int(i/BATCH_SIZE) + 1}/{int(len(all_chunk_paths)/BATCH_SIZE) + 1} with {len(batch_chunk_paths)} files...")

    # Get embeddings for the current batch
    chunk_embed_batch = model.get_audio_embedding_from_filelist(
        x=batch_chunk_paths,
        use_tensor=True
    ).to(DEVICE)

    # Move to CPU and convert to NumPy array to save memory
    chunk_embed_numpy = chunk_embed_batch.cpu().numpy()

    all_chunk_embeddings.append(chunk_embed_numpy)

    # Optional: Clear VRAM/RAM after each batch
    del chunk_embed_batch
    torch.cuda.empty_cache() # Clear unused GPU memory

# Concatenate all batch embeddings into a single NumPy array
final_chunk_audio_embed = np.concatenate(all_chunk_embeddings, axis=0)

# Final Verification
print("\n--- Final Chunk Embeddings Results ---")
print(f"Final chunk embeddings shape: {final_chunk_audio_embed.shape}")
print("Last 20 dimensions of the first chunk embedding:")
print(final_chunk_audio_embed[0, -20:])

Processing chunk batch 1/725 with 10 files...
Processing chunk batch 2/725 with 10 files...
Processing chunk batch 3/725 with 10 files...
Processing chunk batch 4/725 with 10 files...
Processing chunk batch 5/725 with 10 files...
Processing chunk batch 6/725 with 10 files...
Processing chunk batch 7/725 with 10 files...
Processing chunk batch 8/725 with 10 files...
Processing chunk batch 9/725 with 10 files...
Processing chunk batch 10/725 with 10 files...
Processing chunk batch 11/725 with 10 files...
Processing chunk batch 12/725 with 10 files...
Processing chunk batch 13/725 with 10 files...
Processing chunk batch 14/725 with 10 files...
Processing chunk batch 15/725 with 10 files...
Processing chunk batch 16/725 with 10 files...
Processing chunk batch 17/725 with 10 files...
Processing chunk batch 18/725 with 10 files...
Processing chunk batch 19/725 with 10 files...
Processing chunk batch 20/725 with 10 files...
Processing chunk batch 21/725 with 10 files...
Processing chunk batch

**Reasoning**:
The subtask requires saving the chunk metadata and their embeddings. This step will save `all_chunk_metadata` as a CSV file and `final_chunk_audio_embed` as a NumPy file to the specified output directory, making them persistently available for later use.



In [27]:
import pandas as pd
import numpy as np
import os

# Ensure output_dir is defined
output_dir = "/content/drive/MyDrive/Data_audio"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save all_chunk_metadata as a CSV file
chunk_metadata_df = pd.DataFrame(all_chunk_metadata)
chunk_metadata_df.to_csv(os.path.join(output_dir, "all_chunk_metadata.csv"), index=False)
print(f"Saved all_chunk_metadata to {os.path.join(output_dir, 'all_chunk_metadata.csv')}")

# Save chunk embeddings as a NumPy file
np.save(os.path.join(output_dir, "chunk_embeddings.npy"), final_chunk_audio_embed)
print(f"Saved chunk embeddings to {os.path.join(output_dir, 'chunk_embeddings.npy')}")

Saved all_chunk_metadata to /content/drive/MyDrive/Data_audio/all_chunk_metadata.csv
Saved chunk embeddings to /content/drive/MyDrive/Data_audio/chunk_embeddings.npy


**Reasoning**:
The subtask requires updating the Annoy index creation to use the newly generated chunk embeddings. This step will initialize a new Annoy index with the `final_chunk_audio_embed` and save it to a file.



In [28]:
from annoy import AnnoyIndex
import os

# Ensure output_dir is defined
output_dir = "/content/drive/MyDrive/Data_audio"

# Get the dimensionality of the chunk embeddings
f_chunk = final_chunk_audio_embed.shape[1]

# Create an Annoy index for chunk embeddings, using 'angular' for cosine similarity
t_chunk = AnnoyIndex(f_chunk, 'angular')

print("Adding chunk embeddings to Annoy index...")
for i, vec in enumerate(final_chunk_audio_embed):
    t_chunk.add_item(i, vec)

# Build the index (more trees means higher precision, but slower build time)
num_trees = 10 # You can adjust this value
t_chunk.build(num_trees)

# Save the index to a file
annoy_chunk_index_path = os.path.join(output_dir, "audio_chunk_embeddings.ann")
t_chunk.save(annoy_chunk_index_path)

print(f"Annoy index for chunk embeddings created and saved to {annoy_chunk_index_path}")

Adding chunk embeddings to Annoy index...
Annoy index for chunk embeddings created and saved to /content/drive/MyDrive/Data_audio/audio_chunk_embeddings.ann


## Clean up temporary files (optional)

### Subtask:
Add a step to remove the temporary chunked audio files after embeddings have been extracted and saved, to save disk space.


## Summary:

### Q&A
1.  **How many audio chunks were generated from the original files?**
    From 108 original audio files, 7241 10-second audio chunks were generated.
2.  **What is the dimensionality of the generated CLAP embeddings for the audio chunks?**
    Each of the 7241 audio chunks has a 512-dimensional embedding, resulting in a final embedding array shape of (7241, 512).

### Data Analysis Key Findings
*   The `pydub` library was successfully installed and utilized for audio processing.
*   A robust `chunk_audio_file` Python function was created to split audio files into 10-second segments, saving them to a temporary directory and returning metadata (original filename, start, and end times) for each chunk.
*   The existing audio processing workflow was successfully modified to integrate the chunking function, resulting in the generation of 7241 audio chunks from 108 original audio files.
*   CLAP embeddings were successfully generated for all 7241 audio chunks using a batch processing approach to manage memory, yielding a `(7241, 512)` NumPy array.
*   The collected chunk metadata was saved as a CSV file (`all_chunk_metadata.csv`), and the generated chunk embeddings were saved as a NumPy file (`chunk_embeddings.npy`) in the `/content/drive/MyDrive/Data_audio` directory.
*   An Annoy index, built with 10 trees using the 'angular' metric, was successfully created and saved (`audio_chunk_embeddings.ann`) for efficient similarity search on the 512-dimensional chunk embeddings.

### Insights or Next Steps
*   The chunk-level embeddings and their associated metadata provide a more granular way to search and analyze audio content, enabling identification of specific moments within longer audio files rather than just entire files.
*   The Annoy index for chunk embeddings can now be leveraged to perform fast similarity searches, allowing users to find similar 10-second audio segments based on their acoustic properties.


### Searching with Annoy Index for Audio Chunks

This cell allows you to search for audio chunks similar to a text query using the Annoy index created from the chunk embeddings. It will return the nearest chunks along with their original file and time range.

In [ ]:
from annoy import AnnoyIndex
import torch
import numpy as np
import os
import pandas as pd

# Load the Annoy index for chunk embeddings
output_dir = "/content/drive/MyDrive/Data_audio"
annoy_chunk_index_path = os.path.join(output_dir, "audio_chunk_embeddings.ann")

f_chunk = final_chunk_audio_embed.shape[1] # Dimensionality of the chunk embeddings
t_chunk = AnnoyIndex(f_chunk, 'angular') # 'angular' distance is used for cosine similarity
t_chunk.load(annoy_chunk_index_path)
print(f"Annoy index for chunk embeddings loaded from {annoy_chunk_index_path}")

# Load chunk metadata to link indices back to original files and time ranges
chunk_metadata_df = pd.read_csv(os.path.join(output_dir, "all_chunk_metadata.csv"))
print("Chunk metadata loaded.")

# Function to get text embedding (reusing the existing model and function)
def get_text_embedding_with_tensor(text_data):
    # Ensure the model is on the correct device before getting embeddings
    # DEVICE variable should be defined from previous cells ('cuda' or 'cpu')
    text_embed = model.get_text_embedding([text_data], use_tensor=True).to(DEVICE)
    return text_embed


# Interactive loop for searching
print("\nEnter text queries to find nearest audio *chunks* (type 'exit' to quit):\n")
while True:
    text_query = input("Enter text: ")
    if text_query.lower() == 'exit':
        break

    # Get text embedding
    text_embed_tensor = get_text_embedding_with_tensor(text_query)

    # Annoy expects a numpy array for querying, so convert the text embedding
    text_embed_np = text_embed_tensor.cpu().detach().numpy().flatten()

    # Query the Annoy index for the k nearest neighbors
    n_neighbors = 5 # You can adjust this number
    nearest_indices, distances = t_chunk.get_nns_by_vector(text_embed_np, n_neighbors, search_k=-1, include_distances=True)

    print(f"\nNearest {n_neighbors} audio chunks for '{text_query}':")
    for i, idx in enumerate(nearest_indices):
        chunk_info = chunk_metadata_df.iloc[idx]
        original_filename = chunk_info['original_filename']
        start_s = chunk_info['start_s']
        end_s = chunk_info['end_s']
        distance = distances[i]
        print(f"  {i+1}. Original File: {original_filename}, Time Range: {start_s}-{end_s}s (Distance: {distance:.4f})")
    print("\n")

print("Exiting search.")

Annoy index for chunk embeddings loaded from /content/drive/MyDrive/Data_audio/audio_chunk_embeddings.ann
Chunk metadata loaded.

Enter text queries to find nearest audio *chunks* (type 'exit' to quit):

Enter text: piano song

Nearest 5 audio chunks for 'piano song':
  1. Original File: Kim Carroll - The Colony� As it Is (Soundtrack).mp3, Time Range: 60-70s (Distance: 0.9673)
  2. Original File: Moodmusic Radioshow - Echonomist - (13.01.17).mp3, Time Range: 60-70s (Distance: 0.9973)
  3. Original File: Martin Roth - Sad Juno.mp3, Time Range: 10-20s (Distance: 0.9987)
  4. Original File: Kim Carroll - The Colony� As it Is (Soundtrack).mp3, Time Range: 10-20s (Distance: 1.0041)
  5. Original File: Moodmusic Radioshow - Echonomist - (13.01.17).mp3, Time Range: 30-40s (Distance: 1.0151)


Enter text: pure folk guitar

Nearest 5 audio chunks for 'pure folk guitar':
  1. Original File: Message To Bears - At a Glance.mp3, Time Range: 0-10s (Distance: 0.9663)
  2. Original File: Message To Be